# 01. Baselines and Metrics (`TextExperimentAdapter` + `ExperimentTracker`)

In [1]:
from pathlib import Path

import mlflow
import pandas as pd

from meme_organizer.config import MLFLOW_EXPERIMENT_FUSION, MLFLOW_EXPERIMENT_TEXT
from meme_organizer.evaluation.metrics.experiment_tracker import ExperimentTracker
from meme_organizer.evaluation.pipeline.evaluation_dataset_preparer import EvaluationDatasetPreparer
from meme_organizer.retrieval.fusion.hybrid_retriever import HybridRetriever
from meme_organizer.retrieval.text.bm25_retriever import BM25Retriever
from meme_organizer.retrieval.text.dense_text_retriever import DenseTextRetriever
from meme_organizer.retrieval.text.fine_tuned_dense_retriever import FineTunedDenseRetriever
from meme_organizer.retrieval.text.text_experiment_adapter import TextExperimentAdapter

ROOT = Path("..").resolve()
split_data = EvaluationDatasetPreparer().prepare(
    ROOT / "data" / "gold" / "gold_dataset.jsonl",
    ROOT / "data" / "splits" / "splits_v1.json",
    "test",
)
print(f"Corpus: {len(split_data.corpus_docs)} | Test queries: {len(split_data.queries)}")

2026/11/27 22:50:21 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/alex/Projects/meme-organizer/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Corpus: 318 | Test queries: 153


In [2]:
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_TEXT) as exp:
    res_bm25 = exp.create_pipeline(TextExperimentAdapter("bm25_baseline", BM25Retriever(), split_data)).run()
print("BM25:", res_bm25["quality"])

2026/11/27 22:50:24 INFO mlflow.tracking.fluent: Experiment with name 'meme-organizer-text' does not exist. Creating a new experiment.


🏃 View run bm25_baseline at: http://localhost:5000/#/experiments/16/runs/7c501d1f3b8645eaae35265182770e7c
🧪 View experiment at: http://localhost:5000/#/experiments/16
BM25: {'recall_at_1': 0.6078, 'recall_at_5': 0.7255, 'mrr_at_5': 0.6544, 'ndcg_at_5': 0.6722, 'total_queries': 153}


In [3]:
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_TEXT) as exp:
    res_rubert = exp.create_pipeline(
        TextExperimentAdapter(
            "rubert-tiny2_zero_shot", DenseTextRetriever("rubert-tiny2", "cointegrated/rubert-tiny2"), split_data
        )
    ).run()
print("RuBERT:", res_rubert["quality"])

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🏃 View run rubert-tiny2_zero_shot at: http://localhost:5000/#/experiments/16/runs/dc1fe6d931a747f5bc3f39fc1d9617b8
🧪 View experiment at: http://localhost:5000/#/experiments/16
RuBERT: {'recall_at_1': 0.4248, 'recall_at_5': 0.5033, 'mrr_at_5': 0.4517, 'ndcg_at_5': 0.4645, 'total_queries': 153}


In [4]:
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_TEXT) as exp:
    res_e5 = exp.create_pipeline(
        TextExperimentAdapter(
            "e5-small_zero_shot", DenseTextRetriever("e5-small", "intfloat/multilingual-e5-small"), split_data
        )
    ).run()
print("E5:", res_e5["quality"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🏃 View run e5-small_zero_shot at: http://localhost:5000/#/experiments/16/runs/42572ccb05e6461a848a35861bae0b04
🧪 View experiment at: http://localhost:5000/#/experiments/16


E5: {'recall_at_1': 0.549, 'recall_at_5': 0.6667, 'mrr_at_5': 0.5949, 'ndcg_at_5': 0.6129, 'total_queries': 153}
